In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 实验报告及模型说明书 / Experiment Report and Model Documentation

## 1. 实验背景与目的 / Background and Objective

**中文说明：**  
传统计算流体动力学（CFD）的数值模拟通常需要大量的计算资源和时间。随着深度学习技术的发展，利用神经网络来近似求解CFD问题逐渐受到关注。本实验旨在设计并验证一种基于深度学习的混合模型（结合卷积神经网络CNN与全连接神经网络DNN）框架，以加速CFD数值模拟并提高预测精度。

**English Description:**  
Traditional computational fluid dynamics (CFD) simulations often require substantial computational resources and time. With the development of deep learning, neural networks have attracted attention as a means to approximate and accelerate the solution of CFD problems. This experiment aims to design and validate a hybrid deep learning framework (combining CNN and DNN) to speed up CFD simulations and improve prediction accuracy.

---

## 2. 数据集描述 / Dataset Description

**中文说明：**  
本实验使用的公开数据集为 `kaggle-cfd-mixing-tank.csv`，主要包含79列数据，其中：
- **X1**：类别信息（例如 “Design 1”），在本实验中不作为数值特征使用。
- **X2 到 X78**：77个数值特征，描述CFD模拟中的各种参数。
- **Y**：目标变量，代表模拟输出（如流体物理量）。
  
数据总样本数约为1014个，按80%/20%的比例划分为训练集和测试集。

**English Description:**  
The dataset used in this experiment is `kaggle-cfd-mixing-tank.csv`, which contains 79 columns. Among these:
- **X1**: Contains categorical information (e.g., "Design 1"), which is not used as a numerical feature in this experiment.
- **X2 to X78**: 77 numerical features representing various parameters in the CFD simulation.
- **Y**: The target variable representing simulation outputs (e.g., fluid properties).

There are approximately 1014 samples in total, split into training and testing sets with an 80%/20% ratio.

---

## 3. 数据预处理 / Data Preprocessing

**中文说明：**  
1. **数据加载与分割：**  
   - 使用 `pandas.read_csv` 加载数据，指定分隔符为制表符（`\t`）。  
   - 删除 `X1` 列，仅保留数值型特征。
   - 使用 `train_test_split` 将数据划分为训练集和测试集。

2. **特征归一化：**  
   - 使用 `StandardScaler` 对数值特征进行标准化处理（均值为0，标准差为1）。

3. **数据格式转换：**  
   - 对于CNN分支，需将数据转换为形状 `(样本数, 特征数, 1)` 的三维张量；  
   - 对于DNN分支，直接使用二维数据 `(样本数, 特征数)`。

**English Description:**  
1. **Data Loading and Splitting:**  
   - Load the dataset using `pandas.read_csv` with the tab (`\t`) delimiter.  
   - Remove the `X1` column to keep only numerical features.
   - Split the dataset into training and testing sets using `train_test_split`.

2. **Feature Normalization:**  
   - Normalize the numerical features using `StandardScaler` (mean = 0, standard deviation = 1).

3. **Data Reshaping:**  
   - For the CNN branch, reshape the data into a 3D tensor with shape `(number of samples, number of features, 1)`.  
   - For the DNN branch, use the 2D data with shape `(number of samples, number of features)` directly.

---

## 4. 模型构建 / Model Architecture

**中文说明：**  
本实验采用混合模型框架，包含两条分支：  
- **CNN 分支：**  
  - 输入：形状为 `(77, 1)` 的张量  
  - 结构：1D卷积层（64个滤波器，kernel=3） → 最大池化层（pool_size=2） → 1D卷积层（128个滤波器，kernel=3） → 最大池化层（pool_size=2） → Flatten 展平层  
- **DNN 分支：**  
  - 输入：形状为 `(77,)` 的向量  
  - 结构：全连接层 Dense(256, ReLU) → 全连接层 Dense(128, ReLU)  
- **融合输出：**  
  - 将两条支路的输出通过 Concatenate 层合并  
  - 最后通过全连接层 Dense(1, Linear) 得到预测结果

**English Description:**  
The experiment uses a hybrid model architecture with two branches:
- **CNN Branch:**  
  - **Input:** Tensor with shape `(77, 1)`  
  - **Architecture:** 1D Convolutional layer (64 filters, kernel size=3) → MaxPooling1D (pool_size=2) → 1D Convolutional layer (128 filters, kernel size=3) → MaxPooling1D (pool_size=2) → Flatten layer  
- **DNN Branch:**  
  - **Input:** Vector with shape `(77,)`  
  - **Architecture:** Dense layer (256 units, ReLU activation) → Dense layer (128 units, ReLU activation)  
- **Fusion and Output:**  
  - The outputs of both branches are merged using a Concatenate layer  
  - A final Dense layer (1 unit, Linear activation) produces the predicted output

---

## 5. 模型训练 / Model Training

**中文说明：**  
- **优化器：** Adam  
- **损失函数：** 均方误差（MSE）  
- **评估指标：** 均绝对误差（MAE）  
- **训练参数：** 批量大小 32，训练轮数 20，训练集内部再划分10%为验证集

**English Description:**  
- **Optimizer:** Adam  
- **Loss Function:** Mean Squared Error (MSE)  
- **Evaluation Metric:** Mean Absolute Error (MAE)  
- **Training Parameters:** Batch size of 32, 20 epochs, and 10% of the training data used as validation set

---

## 6. 实验结果 / Experimental Results

**中文说明：**  
- 测试集损失 (MSE)：约 0.0017077  
- 测试集MAE：约 0.02822  
此外，通过可视化模块可以观察到：  
- 训练和验证期间损失及MAE的变化曲线  
- 测试集上真实值与预测值的散点图  
- 残差分布直方图

**English Description:**  
- Test set loss (MSE): Approximately 0.0017077  
- Test set MAE: Approximately 0.02822  
Additionally, visualization modules illustrate:  
- The loss and MAE curves during training and validation  
- Scatter plots of true vs. predicted values on the test set  
- The distribution histogram of residuals

---

## 7. 模型架构图 / Model Architecture Diagram

**中文说明：**  
下图为混合模型的架构图，展示了CNN分支与DNN分支的结构及其融合方式。

**English Description:**  
The diagram below illustrates the hybrid model architecture, showing both the CNN branch and the DNN branch along with their fusion.

```python
# 如果环境中没有安装 graphviz，可以先安装：!pip install graphviz
from graphviz import Digraph

dot = Digraph(comment='Hybrid Model Architecture', format='png')

# CNN 分支节点 / CNN Branch
dot.node("A", "Input\n(77, 1)\n(CNN Branch)")
dot.node("B", "Conv1D(64,\nkernel=3,\nReLU)")
dot.node("C", "MaxPooling1D\n(pool=2)")
dot.node("D", "Conv1D(128,\nkernel=3,\nReLU)")
dot.node("E", "MaxPooling1D\n(pool=2)")
dot.node("F", "Flatten")

# DNN 分支节点 / DNN Branch
dot.node("G", "Input\n(77, )\n(DNN Branch)")
dot.node("H", "Dense(256,\nReLU)")
dot.node("I", "Dense(128,\nReLU)")

# 融合与输出 / Fusion and Output
dot.node("J", "Concatenate")
dot.node("K", "Dense(1,\nLinear)\nOutput")

# 构建 CNN 分支的边 / CNN Branch Connections
dot.edge("A", "B")
dot.edge("B", "C")
dot.edge("C", "D")
dot.edge("D", "E")
dot.edge("E", "F")

# 构建 DNN 分支的边 / DNN Branch Connections
dot.edge("G", "H")
dot.edge("H", "I")

# 融合两个分支 / Fuse the Two Branches
dot.edge("F", "J")
dot.edge("I", "J")

# 输出层 / Output Layer
dot.edge("J", "K")

# 渲染并显示图形
dot.render("hybrid_model_architecture", view=True)
dot


8. 实验分析与讨论 / Analysis and Discussion
中文说明：

模型效果： 混合模型在测试集上取得了较低的MSE和MAE，证明了该方法在CFD模拟加速中的有效性。
分支优势： CNN分支能提取局部特征，而DNN分支捕获全局信息，两者结合提升了预测精度。
改进空间： 后续可尝试更深的网络结构、残差连接或注意力机制来进一步优化模型性能。
English Description:

Model Performance: The hybrid model achieved low MSE and MAE on the test set, indicating its effectiveness in accelerating CFD simulations.
Branch Advantages: The CNN branch extracts local features, while the DNN branch captures global information. The combination of both enhances prediction accuracy.
Potential Improvements: Future work may include exploring deeper architectures, residual connections, or attention mechanisms to further optimize performance.
9. 结论与未来工作 / Conclusion and Future Work
中文说明：
本实验展示了基于深度学习的混合模型在CFD数值模拟中的应用潜力。通过结合CNN和DNN两种结构，不仅提高了预测精度，也为加速数值模拟提供了新的思路。未来可以进一步优化网络结构，并扩展到其他CFD问题上。

English Description:
This experiment demonstrates the potential of a hybrid deep learning model in CFD simulations. By combining CNN and DNN architectures, the model improves prediction accuracy and offers a new approach to accelerate numerical simulations. Future work may involve further optimizing the network architecture and extending the approach to other CFD problems.

10. 参考文献 / References
Goodfellow, I., Bengio, Y., & Courville, A. (2016). Deep Learning. MIT Press.
Versteeg, H., & Malalasekera, W. (2007). An Introduction to Computational Fluid Dynamics: The Finite Volume Method. Pearson Education.
OpenFOAM: The Open Source CFD Toolbox. www.openfoam.com

In [ ]:
# 导入必要的库
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, Input

# 打印Kaggle输入目录下的所有文件，确认数据集路径
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# 数据集文件路径（根据你的目录结构进行修改）
file_path = '/kaggle/input/computational-fluid-dynamics-mixing-tank/kaggle-cfd-mixing-tank.csv'

# 读取数据，注意分隔符为制表符
data = pd.read_csv(file_path, delimiter='\t')
print("原始数据预览：")
print(data.head())

# 删除类别信息的列（例如 X1 列，包含 'Design 1' 信息），因为我们只关注数值特征
data.drop(columns=['X1'], inplace=True)

# 分离特征和目标变量（Y列）
X = data.drop(columns=['Y'])
y = data['Y']

# 标准化特征数据
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 为两条分支准备输入数据：
# 1. CNN支路需要3D数据： (样本数, 特征数, 1)
# 2. DNN支路直接使用2D数据： (样本数, 特征数)
X_cnn = X_scaled.reshape(-1, X_scaled.shape[1], 1)  # 用于CNN
X_dnn = X_scaled.copy()                           # 用于DNN

# 保证训练与测试集对应相同的索引：先对2D数据进行分割，再构造CNN支路数据
X_train_dnn, X_test_dnn, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
X_train_cnn = X_train_dnn.reshape(-1, X_train_dnn.shape[1], 1)
X_test_cnn = X_test_dnn.reshape(-1, X_test_dnn.shape[1], 1)

print("训练集样本数：", X_train_dnn.shape[0])
print("测试集样本数：", X_test_dnn.shape[0])

# 构建混合模型

# --- CNN分支 ---
cnn_input = Input(shape=(X_train_dnn.shape[1], 1), name='cnn_input')
x1 = layers.Conv1D(64, kernel_size=3, activation='relu')(cnn_input)
x1 = layers.MaxPooling1D(pool_size=2)(x1)
x1 = layers.Conv1D(128, kernel_size=3, activation='relu')(x1)
x1 = layers.MaxPooling1D(pool_size=2)(x1)
x1 = layers.Flatten()(x1)

# --- DNN分支 ---
dnn_input = Input(shape=(X_train_dnn.shape[1],), name='dnn_input')
x2 = layers.Dense(256, activation='relu')(dnn_input)
x2 = layers.Dense(128, activation='relu')(x2)

# --- 混合两个分支 ---
combined = layers.concatenate([x1, x2])
output = layers.Dense(1, activation='linear')(combined)

# 构建最终模型
model = tf.keras.Model(inputs=[cnn_input, dnn_input], outputs=output)

# 编译模型
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# 输出模型结构
model.summary()

# 训练模型
history = model.fit(
    [X_train_cnn, X_train_dnn],
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# 在测试集上评估模型
loss, mae = model.evaluate([X_test_cnn, X_test_dnn], y_test, verbose=1)
print("Test Loss (MSE):", loss)
print("Test MAE:", mae)


In [ ]:
import matplotlib.pyplot as plt

# 绘制损失和MAE曲线
plt.figure(figsize=(12, 5))

# 损失（MSE）曲线
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss (MSE)')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

# MAE曲线
plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Model MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 对测试集进行预测
y_pred = model.predict([X_test_cnn, X_test_dnn])

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Ideal Fit')  # 对角线
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('True vs Predicted Values')
plt.legend()
plt.show()


In [ ]:
# 计算残差
residuals = y_test - y_pred.flatten()

plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=20, edgecolor='k', alpha=0.7)
plt.title('Distribution of Residuals')
plt.xlabel('Residual')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# 如果环境中没有安装 graphviz，可以先安装
# !pip install graphviz

from graphviz import Digraph

dot = Digraph(comment='Hybrid Model Architecture', format='png')

# CNN 分支节点
dot.node("A", "Input\n(77, 1)\n(CNN Branch)")
dot.node("B", "Conv1D(64,\nkernel=3,\nReLU)")
dot.node("C", "MaxPooling1D(pool=2)")
dot.node("D", "Conv1D(128,\nkernel=3,\nReLU)")
dot.node("E", "MaxPooling1D(pool=2)")
dot.node("F", "Flatten")

# DNN 分支节点
dot.node("G", "Input\n(77, )\n(DNN Branch)")
dot.node("H", "Dense(256,\nReLU)")
dot.node("I", "Dense(128,\nReLU)")

# 合并与输出节点
dot.node("J", "Concatenate")
dot.node("K", "Dense(1,\nLinear)\nOutput")

# 构建 CNN 分支的边
dot.edge("A", "B")
dot.edge("B", "C")
dot.edge("C", "D")
dot.edge("D", "E")
dot.edge("E", "F")

# 构建 DNN 分支的边
dot.edge("G", "H")
dot.edge("H", "I")

# 将两个分支合并
dot.edge("F", "J")
dot.edge("I", "J")

# 最后输出层
dot.edge("J", "K")

# 渲染并显示图形（Kaggle Notebook中会显示图形）
dot.render("hybrid_model_architecture", view=True)
dot
